# Week 3 — Tree Ensembles & Boosting Showdown

**Theme:** Supervised learning II — decision trees, ensembles, and boosting

Last week we used k-NN and linear regression. This week we build up a whole
**family** of tree-based models, in the order they historically appeared and
build on each other:

1. **Decision Tree** — one tree of yes/no questions
2. **Random Forest** — many trees, trained independently, that vote together
   (*bagging*)
3. **AdaBoost** — many *weak* trees, trained one after another, each focusing
   on the previous one's mistakes (*boosting*)
4. **Gradient Boosting (GBM)** — boosting generalized: each new tree fits the
   *residual error* of the ensemble so far
5. **XGBoost** — a heavily optimized, regularized gradient boosting library
   (the long-time favorite for tabular data competitions)
6. **LightGBM** — an even faster gradient boosting library, using a different
   tree-growth strategy

We'll start on scikit-learn's built-in **Breast Cancer Wisconsin** dataset to
see the first two ideas (bagging) cleanly, then switch to a real, messier,
imbalanced **open dataset from Kaggle** for the boosting family, where the
extra power of these algorithms actually starts to matter.

In [ ]:
!pip install -q xgboost lightgbm kagglehub

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## A small helper: `evaluate_model()`

Every model below gets trained and scored the exact same way — fit, time it,
predict, time it, compute accuracy. We write that once and reuse it six
times, appending each result to a running `results` list so we can build one
big comparison table right before the capstone.

In [ ]:
results = []

def evaluate_model(name, dataset, model, X_train, y_train, X_test, y_test):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = model.predict(X_test)
    inference_time = time.perf_counter() - t0

    acc = accuracy_score(y_test, preds)
    results.append({
        "model": name,
        "dataset": dataset,
        "accuracy": acc,
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
    })
    print(f"[{name}] accuracy={acc:.3f}  train={train_time:.3f}s  inference={inference_time:.4f}s")
    return model

## Part 1 — Bagging, on Breast Cancer Wisconsin

scikit-learn ships this dataset built in: 569 tumor samples, 30 numeric
measurements per sample (cell size, texture, etc.), and a binary label
(malignant/benign). Small, clean, well-balanced — perfect for seeing decision
trees and random forests without any data-wrangling distractions.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
print("Features:", len(data.feature_names))
print("Classes:", list(data.target_names))
print("Shape:", X_bc.shape)
print("Class balance:", np.bincount(y_bc))

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=42, stratify=y_bc
)

## 1. Decision Tree

**핵심 하이퍼파라미터 (모델을 만들 때 미리 정해줘야 하는 값들)**

- `criterion` — 어떤 기준으로 "가장 좋은 질문(분할)"을 고를지. 후보: `"gini"`,
  `"entropy"`, `"log_loss"`. **기본값: `"gini"`**
- `max_depth` — 트리가 내려갈 수 있는 최대 깊이. 얕을수록 단순(과소적합 위험),
  깊을수록 복잡(과대적합 위험). 후보: 양의 정수 또는 `None`(제한 없음).
  **기본값: `None`**
- `min_samples_split` — 한 노드를 더 나누기 위해 필요한 최소 샘플 수. 후보:
  정수(개수) 또는 0~1 사이 실수(비율). **기본값: `2`**
- `min_samples_leaf` — 리프(끝마디)에 남아야 하는 최소 샘플 수. 값이 클수록
  더 단순한(덜 과대적합된) 트리가 됨. **기본값: `1`**
- `max_features` — 분할을 고를 때 고려할 특성(feature) 개수. 후보: `"sqrt"`,
  `"log2"`, 정수/실수, 또는 `None`(전체 사용). **기본값: `None`**
- `class_weight` — 클래스 불균형을 보정할지. 후보: `None`, `"balanced"`, 또는
  직접 지정한 딕셔너리. **기본값: `None`**

여기서는 트리를 시각화하기 쉽도록 `max_depth=3`으로 일부러 얕게 만듭니다.

In [ ]:
tree = evaluate_model(
    "Decision Tree", "Breast Cancer",
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_bc_train, y_bc_train, X_bc_test, y_bc_test,
)

plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=data.feature_names, class_names=data.target_names,
          filled=True, fontsize=8)
plt.title("Decision Tree (max_depth=3)")
plt.show()

## 2. Random Forest (bagging)

한 그루의 트리는 데이터가 조금만 바뀌어도 결과가 크게 흔들릴 수 있습니다 (분산이
큼). **랜덤 포레스트**는 데이터와 특성을 각각 무작위로 조금씩 다르게 뽑아 **여러
그루의 트리를 독립적으로** 학습시킨 뒤, 다수결(분류) 또는 평균(회귀)으로 최종
예측을 냅니다 — 이 방식을 **배깅(bagging, bootstrap aggregating)**이라 부릅니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리를 몇 그루 만들지. 많을수록 보통 더 안정적이지만 느려짐.
  후보: 양의 정수. **기본값: `100`**
- `max_depth` — 트리 하나하나의 최대 깊이 (Decision Tree와 동일 의미).
  **기본값: `None`**
- `max_features` — 각 분할에서 고려할 특성 개수 (트리마다 무작위로 다르게 뽑는
  핵심 아이디어). 후보: `"sqrt"`, `"log2"`, 정수/실수, `None`.
  **기본값: `"sqrt"`**
- `min_samples_leaf` — Decision Tree와 동일 의미. **기본값: `1`**
- `bootstrap` — 각 트리를 학습시킬 때 데이터를 복원추출(bootstrap)할지. 후보:
  `True`/`False`. **기본값: `True`**
- `class_weight` — Decision Tree와 동일. **기본값: `None`**

In [ ]:
forest = evaluate_model(
    "Random Forest", "Breast Cancer",
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_bc_train, y_bc_train, X_bc_test, y_bc_test,
)

importances = pd.Series(forest.feature_importances_, index=data.feature_names)
importances.sort_values(ascending=False).head(10).plot(kind="barh", figsize=(6, 4))
plt.title("Top 10 Most Important Features (Random Forest)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Part 2 — Boosting, on a real open dataset

Decision Tree와 Random Forest는 트리들을 **독립적으로** 학습시켰습니다
(bagging). 지금부터 배울 **AdaBoost / Gradient Boosting / XGBoost /
LightGBM**은 전부 **부스팅(boosting)** 계열입니다: 트리를 하나씩 **순서대로**
학습시키면서, 매번 "이전까지의 앙상블이 틀린 부분"에 집중합니다.

이제부터는 좀 더 실감나는 실제 데이터로 넘어갑니다 — Kaggle의
**[Calorie Burn Efficiency 데이터셋](https://www.kaggle.com/datasets/parasharmanu/close-to-realistic-calorie-efficiency-dataset)**:
사람들의 활동/신체 지표로 "칼로리 소모 효율(`calorie_efficiency`)"이 낮음(Low)
/ 보통(Moderate) / 높음(High) 중 무엇인지 맞히는 **3-클래스 분류** 문제입니다.
실제 데이터답게 클래스가 심하게 불균형해서 (대부분 Low), 다루기가 더
흥미롭습니다.

### 데이터 내려받기

`kagglehub`로 바로 내려받습니다 (공개 데이터셋이라 별도 로그인 없이 동작합니다).

In [ ]:
import os
import kagglehub

dataset_dir = kagglehub.dataset_download(
    "parasharmanu/close-to-realistic-calorie-efficiency-dataset"
)
csv_path = os.path.join(dataset_dir, "calorie_efficiency_dataset.csv")
print("Downloaded to:", csv_path)

In [ ]:
df_raw = pd.read_csv(csv_path)
print(df_raw.shape)
df_raw.head()

### 컬럼이 실제로 무엇을 의미하는지

데이터셋 제작자가 공개한 생성 방식에 따르면, 이 데이터는 다음 과정으로
만들어졌습니다:

1. **원시 비율**: `calories_burned / (steps_per_day + 20 × active_minutes)`
   에서 시작 (활동 시간이 걸음 수보다 20배 더 크게 반영됨 — 양보다 강도가
   중요하다는 의미)
2. 여기에 **가중치**를 더합니다: `muscle_mass_ratio`(+0.3, 가장 큰 긍정
   요인), `body_fat_percentage`(−0.2), `hydration_liters`(+0.05),
   `sleep_hours`(+0.05)
3. **심박수 보정**: `80 / heart_rate_resting` (낮을수록 좋음),
   `120 / heart_rate_avg` (너무 높으면 나쁨)
4. **연속 운동일수(`continuous_exercise_days`, 0~7) 보정**: 하루당 +3%,
   5일 이상이면 +10% 추가 보너스 — 단 6일 이상인데 `sleep_hours` < 6이면
   −10% 페널티 (회복 없는 무리한 연속 운동)
5. **하드 컷**: `workouts_per_week` > 6이면 −15% (과훈련), `sleep_hours` < 5면
   −25%
6. 최종적으로 0~10 사이로 정규화한 값이 바로 **`efficiency_score`**이고,
   `calorie_efficiency`는 이 점수를 **고정 구간**으로 나눈 것입니다
   (> 7 High / 4~7 Moderate / < 4 Low) — 단, 그중 **약 8%는 일부러 라벨을
   무작위로 섞어** 현실적인 노이즈를 흉내냈습니다.
7. `bmi`는 이 공식에 직접 쓰이진 않지만 참고용 특징으로 함께 제공됩니다.

즉 `efficiency_score`를 제외한 12개 특징은 전부 타깃을 만드는 **입력값**이라
정상적으로 학습해야 할 특징이지만, `efficiency_score` 자신은 타깃을 만드는
**마지막 단계의 값**이라 사실상 답에 가깝습니다 — 8%의 무작위 노이즈만 빼면요.

### `efficiency_score`가 실제로 얼마나 답에 가까운지 확인

클래스별 분포를 직접 보면, 뚜렷하게 갈리되 (~8% 정도의) 예외도 함께 보일
것입니다 — 바로 그 의도된 라벨 노이즈입니다.

In [ ]:
print(df_raw.groupby("calorie_efficiency")["efficiency_score"].describe())

df_raw.boxplot(column="efficiency_score", by="calorie_efficiency", figsize=(6, 4))
plt.title("efficiency_score by calorie_efficiency class")
plt.suptitle("")
plt.ylabel("efficiency_score")
plt.show()

거의 완벽하게 갈리는 것을 확인했으니, `efficiency_score`는 **정답을 거의
그대로 담고 있는 값(data leakage)**으로 보고 제거합니다 — 그래야 모델이
"점수를 그대로 구간으로 되돌리는" 트릭 대신, 12개의 진짜 행동/신체 지표에서
비선형 상호작용을 학습하게 됩니다 (이게 바로 이번 주차에서 tree 계열 모델을
쓰는 이유입니다).

클래스 분포를 보면 대부분 `Low Efficiency`인 매우 불균형한 데이터이기도
합니다. 이번 주차에서는 (7주차에서 배울 정밀한 불균형 처리 대신) 간단히
**클래스별로 같은 개수만큼 뽑아서** 균형 잡힌 부분집합으로 실습합니다.

In [ ]:
TARGET = "calorie_efficiency"

df = df_raw.drop(columns=["efficiency_score"])

print("Class distribution (raw):")
print(df[TARGET].value_counts())

# Feature engineering: a few interaction ratios (same idea as Week 1's pandas practice)
df["activity_intensity"] = df["active_minutes"] / (df["steps_per_day"] + 1)
df["fitness_ratio"] = df["muscle_mass_ratio"] / (df["body_fat_percentage"] + 1e-5)
df["recovery_score"] = df["sleep_hours"] * df["hydration_liters"]
df["cardio_efficiency"] = df["heart_rate_resting"] / (df["heart_rate_avg"] + 1)

# Balance the classes by downsampling every class to the size of the smallest one
# (capped at 500/class so training stays fast).
per_class = min(df[TARGET].value_counts().min(), 500)
df_balanced = pd.concat(
    [group.sample(per_class, random_state=42) for _, group in df.groupby(TARGET)],
    ignore_index=True,
)
print("\nClass distribution (balanced subset):")
print(df_balanced[TARGET].value_counts())

label_encoder = LabelEncoder()
X_cal = df_balanced.drop(columns=[TARGET])
y_cal = label_encoder.fit_transform(df_balanced[TARGET])
print("\nLabel mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

X_cal_train, X_cal_test, y_cal_train, y_cal_test = train_test_split(
    X_cal, y_cal, test_size=0.3, random_state=42, stratify=y_cal
)

## 3. AdaBoost (Adaptive Boosting)

첫 번째 부스팅 알고리즘입니다. 얕은 트리(보통 깊이 1의 "stump")를 하나씩
학습시키되, **직전 트리가 틀린 샘플의 가중치를 높여서** 다음 트리가 그 부분에
더 집중하게 만듭니다. 최종 예측은 각 트리의 (성능에 따라 가중된) 투표입니다.

**핵심 하이퍼파라미터**

- `estimator` — 부스팅할 약한 학습기(weak learner). 후보: 어떤 분류기든 가능,
  보통 얕은 트리. **기본값: `None`** → 내부적으로 깊이 1짜리
  `DecisionTreeClassifier` (decision stump) 사용
- `n_estimators` — 순차적으로 몇 개의 약한 학습기를 쌓을지. 후보: 양의 정수.
  **기본값: `50`**
- `learning_rate` — 각 학습기의 기여도를 얼마나 줄여서 반영할지 (작을수록
  보수적, `n_estimators`를 늘려 보완해야 함). 후보: 양의 실수. **기본값: `1.0`**

In [ ]:
ada = evaluate_model(
    "AdaBoost", "Calorie Efficiency",
    AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42),
    X_cal_train, y_cal_train, X_cal_test, y_cal_test,
)

## 4. Gradient Boosting (GBM)

AdaBoost를 일반화한 아이디어입니다. 매 단계마다 "정답 - 지금까지의 예측"이라는
**잔차(residual)**를 계산하고, 다음 트리는 그 잔차 자체를 예측하도록
학습시킵니다 — 경사하강법(gradient descent)으로 손실을 줄여나가는 것과 같은
원리라 "gradient" boosting이라 부릅니다.

**핵심 하이퍼파라미터**

- `loss` — 최적화할 손실 함수. 후보: `"log_loss"`, `"exponential"`.
  **기본값: `"log_loss"`**
- `learning_rate` — 각 트리의 기여도를 줄이는 축소 계수(shrinkage). 후보: 양의
  실수. **기본값: `0.1`**
- `n_estimators` — 순차적으로 쌓을 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `subsample` — 각 트리를 학습할 때 사용할 데이터 비율 (1.0 미만이면 일종의
  확률적 경사하강 + 정규화 효과). 후보: 0~1 사이 실수. **기본값: `1.0`**
- `max_depth` — 트리 하나하나의 최대 깊이. 후보: 양의 정수. **기본값: `3`**

In [ ]:
gbm = evaluate_model(
    "Gradient Boosting", "Calorie Efficiency",
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    X_cal_train, y_cal_train, X_cal_test, y_cal_test,
)

## 5. XGBoost (eXtreme Gradient Boosting)

Gradient Boosting을 실전용으로 크게 최적화/정규화한 전문 라이브러리입니다.
결측치를 알아서 처리하고, 병렬로 트리를 빠르게 만들고, 과대적합을 막는
정규화 항(`reg_alpha`, `reg_lambda`)을 손실 함수에 직접 포함시킵니다. 오랫동안
정형(tabular) 데이터 경진대회의 표준 도구였습니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `learning_rate` (`eta`) — 축소 계수. 후보: 0~1 사이 실수. **기본값: `0.3`**
- `max_depth` — 트리 최대 깊이. 후보: 양의 정수. **기본값: `6`**
- `subsample` — 트리마다 사용할 데이터 비율. 후보: 0~1 사이 실수.
  **기본값: `1`**
- `colsample_bytree` — 트리마다 사용할 특성 비율. 후보: 0~1 사이 실수.
  **기본값: `1`**
- `reg_lambda` — L2 정규화 강도 (클수록 더 단순한 모델). 후보: 0 이상 실수.
  **기본값: `1`**
- `reg_alpha` — L1 정규화 강도. 후보: 0 이상 실수. **기본값: `0`**

In [ ]:
xgb = evaluate_model(
    "XGBoost", "Calorie Efficiency",
    XGBClassifier(
        n_estimators=100, learning_rate=0.3, max_depth=6,
        eval_metric="mlogloss", random_state=42,
    ),
    X_cal_train, y_cal_train, X_cal_test, y_cal_test,
)

## 6. LightGBM

Microsoft가 만든 또 다른 gradient boosting 라이브러리입니다. XGBoost가 트리를
레벨(깊이) 단위로 균형 있게 키우는 반면, LightGBM은 **손실을 가장 많이
줄이는 리프(leaf)를 골라서 그 방향으로만 먼저 키우는** 전략(leaf-wise
growth)을 써서 보통 더 빠르고, 대용량 데이터에 특히 강합니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `learning_rate` — 축소 계수. 후보: 0~1 사이 실수. **기본값: `0.1`**
- `num_leaves` — 트리 하나당 최대 리프 개수 (leaf-wise 성장의 핵심 파라미터 —
  깊이 대신 이것으로 복잡도를 제어). 후보: 양의 정수. **기본값: `31`**
- `max_depth` — 트리 최대 깊이 (`-1`은 제한 없음, `num_leaves`가 사실상 더
  중요). 후보: 양의 정수 또는 `-1`. **기본값: `-1`**
- `min_child_samples` — 리프 하나에 필요한 최소 샘플 수 (과대적합 방지).
  후보: 양의 정수. **기본값: `20`**

In [ ]:
lgbm = evaluate_model(
    "LightGBM", "Calorie Efficiency",
    LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, random_state=42, verbose=-1),
    X_cal_train, y_cal_train, X_cal_test, y_cal_test,
)

## 전체 요약: 6개 모델 한눈에 비교

주의: 앞의 두 모델(Decision Tree, Random Forest)은 **Breast Cancer**(이진
분류, 균형 데이터)로, 뒤의 네 모델(AdaBoost~LightGBM)은 **Calorie
Efficiency**(3-클래스, 실제 데이터)로 학습했습니다 — 데이터셋이 다르므로
숫자를 1:1로 비교하기보다는, **같은 데이터셋 안에서의 추세** (배깅→부스팅으로
갈수록, 그리고 부스팅 라이브러리들 사이에서 accuracy/학습시간/추론시간이 어떻게
바뀌는지)를 살펴보세요.

In [ ]:
summary = pd.DataFrame(results)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(summary["model"], summary["accuracy"], color="steelblue")
axes[0].set_title("Accuracy by Model")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(summary["model"], summary["inference_time_sec"], color="indianred")
axes[1].set_title("Inference Time by Model")
axes[1].set_ylabel("Seconds (test set)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Try it yourself

1. **Tune one hyperparameter at a time.** Pick one model (e.g. XGBoost) and
   sweep `max_depth` over `[2, 4, 6, 8]` — plot accuracy vs. depth. Where
   does it start overfitting?
2. **Change the balanced sample size.** Try `per_class = 200` vs `800` —
   does accuracy change much? Does training time?
3. **Bagging vs. boosting, head to head.** Re-run Random Forest on the
   *Calorie Efficiency* dataset (instead of Breast Cancer) so it's a fair
   fight against the four boosting models — does it hold up?
4. **Compare libraries, not just accuracy.** For a model you'd have to
   retrain daily on fresh data, would you pick the most accurate model, or
   the fastest one? What if you had 10 million rows instead of 1,500?

---
## 🎯 캡스톤: XGBoost로 주식 종가 예측하기

지금까지 배운 6개 모델 중 가장 강력했던 **XGBoost**를 실전 회귀(regression)
문제에 적용해봅니다: 과거 주가 데이터로 **다음 날 종가**를 예측하는 모델을
만들어보세요.

아래 데이터 수집/피처 엔지니어링 코드는 그대로 실행하면 되고, 여러분은
**모델을 학습시키고, 예측하고, 평가하는 부분**을 직접 작성합니다.

In [ ]:
# 주가 데이터 내려받기 + 피처 엔지니어링 (실행만 하면 됩니다)
!pip install -q yfinance

import yfinance as yf

TICKER = "AAPL"  # 원하는 종목으로 바꿔도 됩니다 (예: "005930.KS" = 삼성전자)

price_df = yf.download(TICKER, period="3y", interval="1d", progress=False, auto_adjust=True)
price_df = price_df[["Close", "Volume"]].dropna()
price_df.columns = ["close", "volume"]

# 기술적 지표 피처: 이동평균, 변동성, 모멘텀, 거래량 변화율
price_df["ma5"] = price_df["close"].rolling(5).mean()
price_df["ma20"] = price_df["close"].rolling(20).mean()
price_df["volatility10"] = price_df["close"].rolling(10).std()
price_df["return1"] = price_df["close"].pct_change()
price_df["return5"] = price_df["close"].pct_change(5)
price_df["volume_change"] = price_df["volume"].pct_change()

# 예측 타깃: "다음 날" 종가
price_df["target_next_close"] = price_df["close"].shift(-1)
price_df = price_df.dropna()

feature_cols = ["close", "volume", "ma5", "ma20", "volatility10", "return1", "return5", "volume_change"]
X_stock = price_df[feature_cols]
y_stock = price_df["target_next_close"]

# 시계열이므로 랜덤 split이 아니라 "과거로 학습, 최근으로 평가"
split_idx = int(len(price_df) * 0.8)
X_stock_train, X_stock_test = X_stock.iloc[:split_idx], X_stock.iloc[split_idx:]
y_stock_train, y_stock_test = y_stock.iloc[:split_idx], y_stock.iloc[split_idx:]
dates_test = price_df.index[split_idx:]

print(f"Train: {len(X_stock_train)} rows, Test: {len(X_stock_test)} rows")
price_df.tail()

### 여러분의 과제

1. `XGBRegressor`를 만들고 `X_stock_train`, `y_stock_train`으로 학습시키세요.
   (힌트: 회귀 문제이므로 `XGBClassifier`가 아니라 `XGBRegressor`를 씁니다.)
2. 학습된 모델로 `X_stock_test`에 대한 종가를 예측하세요.
3. **실제 종가 vs 예측 종가**를 같은 그래프에 그리고, RMSE를 계산해 출력하세요.
   (힌트: `from sklearn.metrics import mean_squared_error`, 이후
   `mean_squared_error(y_true, y_pred) ** 0.5`)
4. **(자유 과제)** `TICKER`를 다른 종목으로 바꾸거나, `target_next_close`를
   "다음 날" 대신 "5일 뒤" 종가로 바꿔서 실험해보세요. 예측이 더 어려워지나요?

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# TODO 1: XGBRegressor를 만들고 학습시키세요.


# TODO 2: X_stock_test에 대해 예측하세요.


# TODO 3: 실제 vs 예측 종가를 그래프로 그리고, RMSE를 계산해 출력하세요.
